In [21]:
import pandas as pd
import numpy as np


In [22]:
filepath = '../data/processed/dataframe_classificado.csv'

df = pd.read_csv(filepath)

print(f"Dimensões do DataFrame: {df.shape}")
df.head()

Dimensões do DataFrame: (286146, 7)


,CONTA,FINANCEIRO,QUANTIDADE,NOME ATIVO,TIPO DE ATIVO,PERFIL DA CARTEIRA,CLASSIFICACAO DO ATIVO
0,27,16278.264089,16.0,CRI CDIE,TITULOS RF PRIVADOS BRASIL,Moderado,Moderado
1,27,25081.618073,25.0,CRA CDIE,TITULOS RF PRIVADOS BRASIL,Moderado,Moderado
2,27,47046.248881,49.0,CRA IPCA,TITULOS RF PRIVADOS BRASIL,Moderado,Moderado
3,27,36331.476685,35.0,RRRP13 IPCA,DEBENTURES BRASIL,Moderado,Moderado
4,27,18995.623640,20.0,EEELA1 IPCA,DEBENTURES BRASIL,Moderado,Moderado


In [23]:
columns_to_drop = ['QUANTIDADE', 'TIPO DE ATIVO', 'NOME ATIVO', 'PERFIL DA CARTEIRA']
df_droped = df.drop(columns_to_drop, axis=1)
df.head()

,CONTA,FINANCEIRO,QUANTIDADE,NOME ATIVO,TIPO DE ATIVO,PERFIL DA CARTEIRA,CLASSIFICACAO DO ATIVO
0,27,16278.264089,16.0,CRI CDIE,TITULOS RF PRIVADOS BRASIL,Moderado,Moderado
1,27,25081.618073,25.0,CRA CDIE,TITULOS RF PRIVADOS BRASIL,Moderado,Moderado
2,27,47046.248881,49.0,CRA IPCA,TITULOS RF PRIVADOS BRASIL,Moderado,Moderado
3,27,36331.476685,35.0,RRRP13 IPCA,DEBENTURES BRASIL,Moderado,Moderado
4,27,18995.623640,20.0,EEELA1 IPCA,DEBENTURES BRASIL,Moderado,Moderado


In [24]:
df_temp = df_droped.copy()
df_temp['FINANCEIRO'] = df_temp['FINANCEIRO'].abs()

print(f"Number of negative values in original FINANCEIRO column: {(df['FINANCEIRO'] < 0).sum()}")
print(f"Number of negative values in df_temp FINANCEIRO column: {(df_temp['FINANCEIRO'] < 0).sum()}")

df_grouped = df_temp.groupby(['CONTA', 'CLASSIFICACAO DO ATIVO'])['FINANCEIRO'].sum().reset_index()

print(f"Shape of grouped DataFrame: {df_grouped.shape}")
df_grouped

Number of negative values in original FINANCEIRO column: 18974
Number of negative values in df_temp FINANCEIRO column: 0
Shape of grouped DataFrame: (9431, 3)


,CONTA,CLASSIFICACAO DO ATIVO,FINANCEIRO
0,27,Moderado,2.993216e+06
1,44,Moderado,1.700028e+06
2,103,Moderado,8.604886e+06
3,285,Moderado,1.339570e+06
4,299,Moderado,5.453470e+05
...,...,...,...
9426,6167502,Moderado,9.283192e+07
9427,6178808,Conservador,2.710112e+06
9428,6178808,Moderado,1.784979e+07
9429,6184909,Conservador,5.002625e+05


In [25]:
# Step 1: Calcular o total financeiro por conta
total_by_account = df_grouped.groupby('CONTA')['FINANCEIRO'].sum().reset_index()
total_by_account.rename(columns={'FINANCEIRO': 'TOTAL_FINANCEIRO'}, inplace=True)

# Step 2: Juntar o total de volta ao DataFrame agrupado
df_with_total = pd.merge(df_grouped, total_by_account, on='CONTA', how='left')

# Step 3: Calcular o percentual
df_with_total['PERCENTUAL'] = df_with_total['FINANCEIRO'] / df_with_total['TOTAL_FINANCEIRO'] * 100

# Step 4: Pivotar o DataFrame
df_result = df_with_total.pivot_table(
    index='CONTA', 
    columns='CLASSIFICACAO DO ATIVO', 
    values='PERCENTUAL', 
    fill_value=0
)

# Garantir as colunas esperadas
required_columns = ['Conservador', 'Moderado', 'Sofisticado']
for col in required_columns:
    if col not in df_result.columns:
        df_result[col] = 0

# Ordenar as colunas
df_result = df_result[required_columns]

# Resetar nomes de eixo para remover "CLASSIFICACAO DO ATIVO"
df_result.columns.name = None  # Remove o nome do eixo de colunas
df_result.index.name = 'Conta'  # Mantém 'Conta' como índice

# Exibir o resultado
print(f"Shape of new DataFrame: {df_result.shape}")
df_result.head()


Shape of new DataFrame: (6101, 3)


,Conservador,Moderado,Sofisticado
Conta,,,
27,0.0,100.0,0.0
44,0.0,100.0,0.0
103,0.0,100.0,0.0
285,0.0,100.0,0.0
299,0.0,100.0,0.0


In [26]:
# Criar novo DataFrame a partir do df_result
df_final = df_result.reset_index()  # Tira 'Conta' do índice e vira coluna normal

# Reorganizar as colunas, só por garantia
df_final = df_final[['Conta', 'Conservador', 'Moderado', 'Sofisticado']]

# Exibir resultado
print(f"Shape of final DataFrame: {df_final.shape}")
df_final.head(100)


Shape of final DataFrame: (6101, 4)


,Conta,Conservador,Moderado,Sofisticado
0,27,0.000000,100.000000,0.000000
1,44,0.000000,100.000000,0.000000
2,103,0.000000,100.000000,0.000000
3,285,0.000000,100.000000,0.000000
4,299,0.000000,100.000000,0.000000
...,...,...,...,...
95,16524,0.000000,100.000000,0.000000
96,16759,89.483516,9.221191,1.295294
97,16800,0.000000,100.000000,0.000000
98,16821,0.000000,100.000000,0.000000


In [27]:
df_final.to_csv('../data/processed/percentual_de_distribuicao.csv', index=False)

In [28]:
df

,CONTA,FINANCEIRO,QUANTIDADE,NOME ATIVO,TIPO DE ATIVO,PERFIL DA CARTEIRA,CLASSIFICACAO DO ATIVO
0,27,1.627826e+04,16.0,CRI CDIE,TITULOS RF PRIVADOS BRASIL,Moderado,Moderado
1,27,2.508162e+04,25.0,CRA CDIE,TITULOS RF PRIVADOS BRASIL,Moderado,Moderado
2,27,4.704625e+04,49.0,CRA IPCA,TITULOS RF PRIVADOS BRASIL,Moderado,Moderado
3,27,3.633148e+04,35.0,RRRP13 IPCA,DEBENTURES BRASIL,Moderado,Moderado
4,27,1.899562e+04,20.0,EEELA1 IPCA,DEBENTURES BRASIL,Moderado,Moderado
...,...,...,...,...,...,...,...
286141,6178808,3.890751e+05,400.0,RSAN16 IPCA,DEBENTURES BRASIL,Sofisticado,Moderado
286142,6178808,4.049765e+05,423.0,CMGDB1 IPCA,DEBENTURES BRASIL,Sofisticado,Moderado
286143,6178808,1.576249e+06,1557.0,EGIEB4 IPCA,DEBENTURES BRASIL,Sofisticado,Moderado
286144,6184909,5.002625e+05,637.0,NTNF PRE,TITULOS RF PUBLICOS BRASIL,Conservador,Conservador


In [29]:
user_item_binary = df.assign(tem_ativo=1).pivot_table(
    index='CONTA',
    columns='NOME ATIVO',  # ou 'CLASSIFICACAO DO ATIVO'
    values='tem_ativo',
    aggfunc='max',
    fill_value=0
)

user_item_binary.head()


NOME ATIVO,1832421D CN,1SANPN,2299955D CN,2458693D US,700 HK,8001 JP,8031 JP,8058 JP,9988 HK,A1AP34,...,YUFI11,Z US.,Z1OM34,ZAGH11,ZAMP3,ZAVI11,ZIFI11,ZS US,ZTO US,ZTS US
CONTA,,,,,,,,,,,,,,,,,,,,,
27,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
44,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
103,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
285,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
299,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [30]:
user_item_binary.to_csv('../data/processed/user_item_binario.csv', index=False)